In [1]:
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt

from UtilsRF import scorer

In [2]:
def print_sentence_relation_subj_obj(data, print_text="predicted"):
    for i, sentence in enumerate(data["token"]):
        print(f"{print_text}: {data.iloc[i][print_text]} (id: {data.iloc[i]['id']})")
        print(
            f"Subject: {data.iloc[i]['token'][data.iloc[i]['subj_start']:data.iloc[i]['subj_end']+1]}"
        )
        print(
            f"Object: {data.iloc[i]['token'][data.iloc[i]['obj_start']:data.iloc[i]['obj_end']+1]}"
        )
        print(f"{' '.join(sentence)}\n")

In [3]:
def summary_relation_FN(dataset, relation, summary_data, not_long=True):
    """
    Given a dataset and a relation, it outputs the sentences where that relation is the correct one and the predicted one is wrong (i.e. TP and FN for that relation).
    It provides some summaries (number of wrong relations, predicted relations, number of sentences with long distance between entities).
    Afterwards, it prints the sentences along with the entities and the predicted relations.
    If not_long is True, it only prints the sentences where the distance between entities is NOT long
    """
    data_relation = test_data[dataset["relation"] == relation]
    print(f"Number of sentences with relation {relation} : {len(data_relation)}")

    data_relation_wrong = data_relation[
        data_relation["relation"] != data_relation["predicted"]
    ]
    print(
        f"Number of sentences with wrong relation {relation} : {len(data_relation_wrong)}"
    )

    data_relation_wrong_not_long = data_relation_wrong[
        data_relation_wrong["entity_distance_categorical"] != "long"
    ]
    print(
        f"Of those sentences, {len(data_relation_wrong) - len(data_relation_wrong_not_long)} have long distance between entities"
    )

    wrong_relations_summary = data_relation_wrong["predicted"].value_counts()
    print(
        f"\nSummary of relations assigned instead of {relation}:\n{wrong_relations_summary}\n"
    )

    print(f"List of (NOT LONG) sentences with relation {relation} wrongly predicted")
    print_sentence_relation_subj_obj(
        data_relation_wrong_not_long if not_long else data_relation_wrong,
        print_text="predicted",
    )

In [4]:
def summary_relation_FP(dataset, relation, summary_data, not_long=True):
    """
    Given a dataset and a relation, it outputs the sentences the model outputs the relation while the correct one is different (i.e. TP and FP for that relation).
    It provides some summaries (number of wrong relations, predicted relations, number of sentences with long distance between entities).
    Afterwards, it prints the sentences along with the entities and the predicted relations.
    If not_long is True, it only prints the sentences where the distance between entities is NOT long
    """
    data_relation = test_data[dataset["predicted"] == relation]
    print(
        f"Number of sentences where the model predicts relation {relation} : {len(data_relation)}"
    )

    data_relation_wrong = data_relation[data_relation["relation"] != relation]
    print(
        f"Number of sentences where the model outputs {relation} but the true one is a different one: {len(data_relation_wrong)}"
    )

    data_relation_wrong_not_long = data_relation_wrong[
        data_relation_wrong["entity_distance_categorical"] != "long"
    ]
    print(
        f"Of those sentences, {len(data_relation_wrong) - len(data_relation_wrong_not_long)} have long distance between entities"
    )

    wrong_relations_summary = data_relation_wrong["relation"].value_counts()
    print(
        f"\nSummary of true relations to which {relation} is wrongly assigned:\n{wrong_relations_summary}\n"
    )

    print(f"List of (NOT LONG) sentences with relation {relation} assigned wrongly")
    print_sentence_relation_subj_obj(
        data_relation_wrong_not_long if not_long else data_relation_wrong,
        print_text="relation",
    )

## Relation-specific error analysis.
In this notebook a more detailed breakdown of the most misclassified relations is achieved. In order to do this, we start by focusing on the relations where we saw we had 0 recall.

In [6]:
tacrev_results = pd.read_csv("predictions_tacrev.txt", sep="\t")

In [7]:
summary_data = scorer.score(
    tacrev_results["Gold Label"],
    tacrev_results["Prediction"],
    verbose=True,
    return_data=True,
).rename(
    columns={
        "F1": "Tacred_F1",
        "Precision": "Tacred_precision",
        "Recall": "Tacred_recall",
    }
)
summary_data.rename(
    columns={
        "Relation": "relation",
        "Tacred_precision": "Tacrev_precision",
        "Tacred_recall": "Tacrev_recall",
        "Tacred_F1": "Tacrev_F1",
    },
    inplace=True,
)

Per-relation statistics:
org:alternate_names                  P:  78.20%  R:  67.35%  F1:  72.37%  #: 245
org:city_of_headquarters             P:  68.49%  R:  56.18%  F1:  61.73%  #: 89
org:country_of_headquarters          P:  44.29%  R:  33.70%  F1:  38.27%  #: 92
org:dissolved                        P: 100.00%  R:   0.00%  F1:   0.00%  #: 1
org:founded                          P: 100.00%  R:   0.00%  F1:   0.00%  #: 37
org:founded_by                       P: 100.00%  R:   0.00%  F1:   0.00%  #: 76
org:member_of                        P: 100.00%  R:   0.00%  F1:   0.00%  #: 4
org:members                          P: 100.00%  R:   0.00%  F1:   0.00%  #: 16
org:number_of_employees/members      P: 100.00%  R:   0.00%  F1:   0.00%  #: 14
org:parents                          P:   0.00%  R:   0.00%  F1:   0.00%  #: 57
org:political/religious_affiliation  P:  11.11%  R:  10.00%  F1:  10.53%  #: 10
org:shareholders                     P: 100.00%  R:   0.00%  F1:   0.00%  #: 3
org:stateorprovin

In [13]:
# Load data from JSON file
with open("../../../data/tacred/json/train.json", "r") as file:
    data_train = json.load(file)

# Convert JSON data to DataFrame
training_data = pd.json_normalize(data_train)

# Count occurrences of each relation in the training set
relation_counts = training_data["relation"].value_counts()

# Reset index and rename columns
train_counts_df = relation_counts.reset_index()
train_counts_df.rename(
    columns={"index": "relation", "relation": "train_counts"}, inplace=True
)

# Assuming summary_data has a column named 'relation'
summary_data = pd.merge(summary_data, train_counts_df, on="relation")

In [14]:
summary_data["train_counts"].describe()

count      40.000000
mean      303.350000
std       472.763103
min         6.000000
25%        75.500000
50%       143.500000
75%       305.750000
max      2287.000000
Name: train_counts, dtype: float64

Consider on which relations the lower precision and recall levels are achieved (excluding those with recall 0):

In [15]:
summary_data[summary_data["Tacrev_recall"] != 0].sort_values(
    "Tacrev_precision"
).head()  # Most of these relations are below the mean but far from being the smallest (see distr above)

,relation,Tacrev_precision,Tacrev_recall,Tacrev_F1,test_counts,train_counts
10,org:political/religious_affiliation,0.111111,0.100000,0.105263,10,95
35,per:spouse,0.166667,0.030303,0.051282,66,240
28,per:employee_of,0.279195,0.825397,0.417252,252,1441
21,per:cities_of_residence,0.293651,0.284615,0.289062,130,351
32,per:religion,0.363636,0.093023,0.148148,43,51


In [16]:
# These appear to be quite large for what concerns the training counts, so would have to be investigated.
summary_data[summary_data["Tacrev_recall"] != 0].sort_values("Tacrev_recall").head()

,relation,Tacrev_precision,Tacrev_recall,Tacrev_F1,test_counts,train_counts
35,per:spouse,0.166667,0.030303,0.051282,66,240
19,per:charges,0.666667,0.075472,0.135593,106,68
32,per:religion,0.363636,0.093023,0.148148,43,51
10,org:political/religious_affiliation,0.111111,0.100000,0.105263,10,95
12,org:stateorprovince_of_headquarters,0.647059,0.215686,0.323529,51,221


Note that all the relationships above fall in the category of relations which have a reverse relation or some similar relations (both in terms of logical sense but also as described in the guidelines for the annotations, where the annotators are explicitly told to not confuse certain instances).

For this reason we decided to investigate these groups of relationships further as they also included those relations for which the model has recall 0. When looking into the sentences predicted wrongly directly, we exclude the sentences which fall in the category of having long distance between the subject and object entities.

In [17]:
with open("../../../data/tacrev/json/test.json", "r") as file:
    test_data = json.load(file)
test_data = pd.json_normalize(test_data)

In [18]:
test_data["predicted"] = tacrev_results["Prediction"]
test_data["entity_distance"] = [
    (test_data["subj_start"][i] - test_data["obj_end"][i])
    if test_data["subj_start"][i] > test_data["obj_end"][i]
    else (test_data["obj_start"][i] - test_data["subj_end"][i])
    for i in range(len(test_data))
]
q1_ed = test_data["entity_distance"].quantile(0.3333)
q2_ed = test_data["entity_distance"].quantile(0.6666)
test_data["entity_distance_categorical"] = [
    "short"
    if test_data["entity_distance"][i] < q1_ed
    else ["long" if test_data["entity_distance"][i] > q2_ed else "medium"][0]
    for i in range(len(test_data))
]

- - - 
## ORG: Relations
### Focus on the family of relations relative to membership and relations with other companies/organisations
From these family we consider these types of relations, as they are mentioned as (be careful it is different from) in the slot annotation guidelines.
- `org:member_of` and `org:member` (opposite relations)
- `org:parents` and `org:subsidiaries` (opposite relations)
- `org:shareholders`

First of all extract statistics for these relationships:

In [19]:
membership_relations = [
    "org:member_of",
    "org:members",
    "org:parents",
    "org:subsidiaries",
    "org:shareholders",
]
filtered_summary_data = summary_data[
    summary_data["relation"].isin(membership_relations)
].sort_values("Tacrev_precision", ascending=False)
filtered_summary_data

,relation,Tacrev_precision,Tacrev_recall,Tacrev_F1,test_counts,train_counts
6,org:member_of,1.0,0.0,0.0,4,113
7,org:members,1.0,0.0,0.0,16,155
11,org:shareholders,1.0,0.0,0.0,3,70
9,org:parents,0.0,0.0,0.0,57,268
13,org:subsidiaries,0.0,0.0,0.0,31,278


Let us start by investigate `org:shareholders` and  `org:member_of` and `org:member`  which are never predicted by the model 

In [20]:
summary_relation_FN(test_data, "org:shareholders", summary_data)

Number of sentences with relation org:shareholders : 3
Number of sentences with wrong relation org:shareholders : 3
Of those sentences, 0 have long distance between entities

Summary of relations assigned instead of org:shareholders:
no_relation                  2
org:top_members/employees    1
Name: predicted, dtype: int64

List of (NOT LONG) sentences with relation org:shareholders wrongly predicted
predicted: org:top_members/employees (id: 098f6cd9013bf31021bb)
Subject: ['Sycamore', 'Valley', 'Ranch', 'Co.']
Object: ['Jackson']
Sycamore Valley Ranch Co. is a joint venture between Jackson and an affiliate of Colony Capital LLC , according to person with knowledge of the transaction who was not authorized to speak on the record and requested anonymity .

predicted: no_relation (id: 098f6cd90161b39172fd)
Subject: ['Alico']
Object: ['AIG']
AIG first signaled its intent to offer the government stakes in AIA and Alico on March 2 , the same day that the company posted a staggering $ 62 bil

In [21]:
summary_relation_FP(test_data, "org:shareholders", summary_data)

Number of sentences where the model predicts relation org:shareholders : 0
Number of sentences where the model outputs org:shareholders but the true one is a different one: 0
Of those sentences, 0 have long distance between entities

Summary of true relations to which org:shareholders is wrongly assigned:
Series([], Name: relation, dtype: int64)

List of (NOT LONG) sentences with relation org:shareholders assigned wrongly


There are only 3 sentences, none of which has a large ditance between the subject and the object. The model predicts twice "no_relation" and once "org:top_members/employees" which could be seen as similar to the right prediction. 


In [22]:
summary_relation_FN(test_data, "org:member_of", summary_data)

Number of sentences with relation org:member_of : 4
Number of sentences with wrong relation org:member_of : 4
Of those sentences, 2 have long distance between entities

Summary of relations assigned instead of org:member_of:
no_relation    4
Name: predicted, dtype: int64

List of (NOT LONG) sentences with relation org:member_of wrongly predicted
predicted: no_relation (id: 098f6784e78abd5c3e0c)
Subject: ['ALICO']
Object: ['AIG']
ALICO , a member company of AIG is looking for one J2EE developer to help with a significant re-engineering project .

predicted: no_relation (id: 098f6e00a8db2ff2cd29)
Subject: ['UASR']
Object: ['Muslim', 'Brotherhood', "'s", 'Palestine', 'Committee']
During the trial , FBI agent Lara Burns testified that UASR was part of the Muslim Brotherhood 's Palestine Committee in America < http://www.investigativeproject.org/article/361 > .



In [23]:
summary_relation_FP(test_data, "org:member_of", summary_data)

Number of sentences where the model predicts relation org:member_of : 0
Number of sentences where the model outputs org:member_of but the true one is a different one: 0
Of those sentences, 0 have long distance between entities

Summary of true relations to which org:member_of is wrongly assigned:
Series([], Name: relation, dtype: int64)

List of (NOT LONG) sentences with relation org:member_of assigned wrongly


In this case there are 4 sentences, 2 of which have a long distance between the subject and the object, in these cases the model predicts no_relation and org:parents. 
On the other side the model wrongly assign org:member_of relation to a case where the right prediction is org:parents. It looks like the model is inverting the labels in this case. 

In [24]:
summary_relation_FN(test_data, "org:members", summary_data)

Number of sentences with relation org:members : 16
Number of sentences with wrong relation org:members : 16
Of those sentences, 9 have long distance between entities

Summary of relations assigned instead of org:members:
no_relation                    14
org:country_of_headquarters     1
org:alternate_names             1
Name: predicted, dtype: int64

List of (NOT LONG) sentences with relation org:members wrongly predicted
predicted: no_relation (id: 098f6e00a8a07ffb432f)
Subject: ['Organization', 'of', 'Asia', '-', 'Pacific', 'News', 'Agencies']
Object: ['Azerbaijani', 'wire', 'service']
Trend , an Azerbaijani wire service , on Thursday became a full member of the Organization of Asia - Pacific News Agencies -LRB- OANA -RRB- .

predicted: org:country_of_headquarters (id: 098f6e00a8c213b31b91)
Subject: ['Pacific', 'Asia', 'Travel', 'Association']
Object: ['Taiwan']
The fair was organized by the Pacific Asia Travel Association , of which Taiwan is a senior member .

predicted: org:alter

In [69]:
summary_relation_FP(test_data, "org:members", summary_data)

Number of sentences where the model predicts relation org:members : 17
Number of sentences where the model outputs org:members but the true one is a different one: 17
Of those sentences, 1 have long distance between entities

Summary of true relations to which org:members is wrongly assigned:
relation
no_relation            16
org:alternate_names     1
Name: count, dtype: int64

List of (NOT LONG) sentences with relation org:members assigned wrongly
relation: no_relation (id: 098f665fb9ded45248e8)
Subject: ['Koch', 'Foods']
Object: ['Customs', 'Enforcement']
U.S. Immigration and Customs Enforcement agents seized documents and other materials at the Koch Foods plant in southwest Ohio and at Koch Foods Inc. 's Chicago-area headquarters , said Brian Moskowitz , an agent in charge of ICE enforcement for Ohio and Michigan .

relation: no_relation (id: 098f665fb9df7cd35240)
Subject: ['Koch', 'Foods']
Object: ['Customs', 'Enforcement']
U.S. Immigration and Customs Enforcement agents seized do

The model mainly assign no_relation. In some cases it assigns org:alternate_names when it should assign org:members and viceversa: a reason for these missclassifications could be that the model is not taking into account the context.

In [76]:
summary_relation_FN(test_data, "org:parents", summary_data)

Number of sentences with relation org:parents : 57
Number of sentences with wrong relation org:parents : 53
Of those sentences, 7 have long distance between entities

Summary of relations assigned instead of org:parents:
predicted
no_relation                    46
org:alternate_names             2
org:top_members/employees       2
org:member_of                   1
org:subsidiaries                1
org:country_of_headquarters     1
Name: count, dtype: int64

List of (NOT LONG) sentences with relation org:parents wrongly predicted
predicted: no_relation (id: 098f6f7060adc06158f3)
Subject: ['ALICO']
Object: ['MetLife', 'Inc']
It sold ALICO to MetLife Inc for $ 162 billion .

predicted: no_relation (id: 098f6f70609f424b7b16)
Subject: ['Countrywide', 'Financial', 'Corp.']
Object: ['Bank', 'of', 'America', 'Corp.']
Bank of America Corp. is taking a similar approach with newly acquired Countrywide Financial Corp. as part of an $ 8.4 billion , 12-state legal settlement reached this month .

pr

In [77]:
summary_relation_FP(test_data, "org:parents", summary_data)

Number of sentences where the model predicts relation org:parents : 44
Number of sentences where the model outputs org:parents but the true one is a different one: 40
Of those sentences, 7 have long distance between entities

Summary of true relations to which org:parents is wrongly assigned:
relation
no_relation                  32
org:alternate_names           3
org:member_of                 2
org:subsidiaries              2
org:top_members/employees     1
Name: count, dtype: int64

List of (NOT LONG) sentences with relation org:parents assigned wrongly
relation: no_relation (id: 098f665fb951f23140f6)
Subject: ['National', 'Urban', 'League']
Object: ['Census', 'Advisory', 'Committee']
`` In 2000 , we had an undercount in communities of color , and an overcount in white communities , '' said Marc Morial , president of the National Urban League and chairman of the Census Advisory Committee .

relation: no_relation (id: 098f665fb9e7c193b5dc)
Subject: ['Loose', 'Change']
Object: ['Forum'

With the relation org:parents in many cases it missclassifies the relations because it is not taking into account the context, this is clear when looking at sentences in which it assigns org:alternate_name. In same cases, it assigns org:memeber_of which is a different relation but still very similar. 

In [79]:
summary_relation_FN(test_data, "org:subsidiaries", summary_data)

Number of sentences with relation org:subsidiaries : 31
Number of sentences with wrong relation org:subsidiaries : 27
Of those sentences, 3 have long distance between entities

Summary of relations assigned instead of org:subsidiaries:
predicted
no_relation      24
org:parents       2
org:member_of     1
Name: count, dtype: int64

List of (NOT LONG) sentences with relation org:subsidiaries wrongly predicted
predicted: no_relation (id: 098f6c74c502d16f9b90)
Subject: ['National', 'Energy', 'Administration']
Object: ['New', 'Energy', 'Department']
The country 's installed wind power capacity will reach 20 gigawatts this year , said Shi Lishan , vice director of the National Energy Administration 's New Energy Department , the Xinhua news agency said Wednesday .

predicted: no_relation (id: 098f6c74c50acc616c22)
Subject: ['Urban', 'League']
Object: ['Colman', 'School']
The Urban League , which bought the Colman School , is confronting the community issue directly .

predicted: no_relation 

In [80]:
summary_relation_FP(test_data, "org:subsidiaries", summary_data)

Number of sentences where the model predicts relation org:subsidiaries : 22
Number of sentences where the model outputs org:subsidiaries but the true one is a different one: 18
Of those sentences, 3 have long distance between entities

Summary of true relations to which org:subsidiaries is wrongly assigned:
relation
no_relation            16
org:alternate_names     1
org:parents             1
Name: count, dtype: int64

List of (NOT LONG) sentences with relation org:subsidiaries assigned wrongly
relation: no_relation (id: 098f665fb98a41514294)
Subject: ['ADF']
Object: ['Bersama', 'Shield']
Bersama Shield has also provided the ADF with the opportunity to develop relationships with important security partners , while reinforcing Australia 's long-term commitment to regional stability .

relation: no_relation (id: 098f665fb96d22986df7)
Subject: ['American', 'Free', 'Press']
Object: ['Lawrence', 'Livermore', 'National', 'Lab']
`` That 's baloney , '' Marion Fulk , a retired staff scientist 

It's interesting to observe how the model confuses org:subsidiaries and org:parents

- - - 
### Focus on the family of relations relative date of foundation/dissolution of an organisation.

These are:
- `org:founded`
- `org:dissolved`

Note that the latter appears only once in the testing set, so claiming that it has bad F1 would be unwise given the limited number of examples. On the contrary, founded is quite good.

In [71]:
summary_relation_FN(test_data, "org:founded", summary_data)

Number of sentences with relation org:founded : 37
Number of sentences with wrong relation org:founded : 30
Of those sentences, 0 have long distance between entities

Summary of relations assigned instead of org:founded:
predicted
no_relation    30
Name: count, dtype: int64

List of (NOT LONG) sentences with relation org:founded wrongly predicted
predicted: no_relation (id: 098f63858279daf1ef91)
Subject: ['New', 'Fabris']
Object: ['1947']
Founded in 1947 by two brothers , Eugene and Quentin Fabris , New Fabris started out making sewing machine parts , before branching out into the auto sector , employing up to 800 workers in the 1990s .

predicted: no_relation (id: 098f63858251ac14afd5)
Subject: ['National', 'Congress', 'of', 'American', 'Indians']
Object: ['1944']
The National Congress of American Indians was founded in 1944 in response to assimilation policies being imposed on tribes by the federal government .

predicted: no_relation (id: 098f638582fd45e6cd93)
Subject: ['UASR']
Obje

In [72]:
summary_relation_FP(test_data, "org:founded", summary_data)

Number of sentences where the model predicts relation org:founded : 16
Number of sentences where the model outputs org:founded but the true one is a different one: 9
Of those sentences, 0 have long distance between entities

Summary of true relations to which org:founded is wrongly assigned:
relation
no_relation    9
Name: count, dtype: int64

List of (NOT LONG) sentences with relation org:founded assigned wrongly
relation: no_relation (id: 098f665fb9cb453c0c7c)
Subject: ['Pacific', 'Asia', 'Travel', 'Association']
Object: ['Tuesday']
More than 230 tourism executives , government officials and analysts attended the Pacific Asia Travel Association meeting Tuesday and Wednesday , billed as the region 's first to seek practical solutions to climate change .

relation: no_relation (id: 098f665fb9d5639d7619)
Subject: ['Menil', 'Collection']
Object: ['1986']
Since his Menil Collection building opened in Houston in 1986 , Piano 's use of light has inspired fervent admiration .

relation: no_r

In [26]:
summary_relation_FN(test_data, "org:dissolved", summary_data)

Number of sentences with relation org:dissolved : 1
Number of sentences with wrong relation org:dissolved : 1
Of those sentences, 0 have long distance between entities

Summary of relations assigned instead of org:dissolved:
no_relation    1
Name: predicted, dtype: int64

List of (NOT LONG) sentences with relation org:dissolved wrongly predicted
predicted: no_relation (id: 098f6aee5e82b98e0308)
Subject: ['New', 'Fabris']
Object: ['June', '16']
New Fabris closed down June 16 .



In [25]:
summary_relation_FP(test_data, "org:dissolved", summary_data)

Number of sentences where the model predicts relation org:dissolved : 0
Number of sentences where the model outputs org:dissolved but the true one is a different one: 0
Of those sentences, 0 have long distance between entities

Summary of true relations to which org:dissolved is wrongly assigned:
Series([], Name: relation, dtype: int64)

List of (NOT LONG) sentences with relation org:dissolved assigned wrongly
